# 04 - Matrices de Conectividad, Grafos de Vecindad y Escalabilidad en Clustering Jerárquico

## Curso de Machine Learning: Modelos No Supervisados

El mayor obstáculo que enfrenta el clustering jerárquico aglomerativo estándar en entornos de producción y Big Data es su **complejidad computacional**:
- **Memoria**: Requiere almacenar la matriz completa de distancias por pares $n \times n$, lo cual consume $O(n^2)$ de memoria RAM (con $n = 50,000$, la matriz ocupa $\approx 20$ GB).
- **Tiempo de Cómputo**: Buscar iterativamente el par mínimo toma entre $O(n^2 \log n)$ y $O(n^3)$.

¿Cómo podemos acelerar el clustering aglomerativo por órdenes de magnitud y permitir que procese decenas de miles de observaciones? La respuesta de Scikit-Learn consiste en introducir una **Matriz de Conectividad Estructurada (*Connectivity Graph*)** mediante el parámetro `connectivity` en `AgglomerativeClustering`.

Al imponer que **dos clusters únicamente pueden fusionarse si comparten al menos una arista en un grafo de adyacencia espacial o de $k$-vecinos más cercanos ($k$-NN)**, el problema se transforma de una búsqueda cuadrática densa a un recorrido eficiente sobre matrices dispersas (*sparse matrices*).

---

### Contenidos
1. El Cuello de Botella Cuadrático del Clustering Jerárquico Tradicional
2. Concepto de Restricción de Conectividad (*Connectivity Constraints*)
3. Construcción del Grafo de $k$-Vecinos con `kneighbors_graph`
4. Ventajas Topológicas: Preservación de Variedades Curvas y Supresión de Enlaces Espurios
5. Benchmark de Rendimiento: Tiempo de Ejecución y Memoria (Sin Conectividad vs. Con Conectividad)
6. Conclusiones


In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import make_swiss_roll, make_blobs
from sklearn.neighbors import kneighbors_graph
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import adjusted_rand_score

# Configuración visual
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
np.random.seed(42)


## 1. Fundamentos: ¿Qué es la Matriz de Conectividad?

Una matriz de conectividad es una matriz dispersa binaria $C \in \{0, 1\}^{n \times n}$ donde:
$$C_{ij} = \begin{cases} 1 & \text{si la muestra } i \text{ y la muestra } j \text{ son vecinas directas en el grafo} \\ 0 & \text{en caso contrario} \end{cases}$$

Cuando pasamos `AgglomerativeClustering(connectivity=C)`:
1. Dos clusters $A$ y $B$ **solo son elegibles para fusionarse si existe al menos un par de puntos conectado por una arista ($C_{ab} = 1$)**.
2. El algoritmo ignora el $99.5\%$ de las combinaciones lejanas, reduciendo la búsqueda a vecinos locales inmediatos.
3. Además de acelerar el cálculo, **evita que el algoritmo puentee clusters separados que se encuentran cerca en el espacio euclidiano pero pertenecen a ramas o variedades topológicas distintas** (por ejemplo, dos capas adyacentes de un *Swiss Roll* o dos curvas paralelas).


In [ ]:
# Demostración: Estructura de Variedad Curva (Swiss Roll desplegado)
X_swiss, color_swiss = make_swiss_roll(n_samples=1500, noise=0.05, random_state=42)
# Tomamos una sección 2D para visualización clara
X_2d = np.column_stack([X_swiss[:, 0], X_swiss[:, 2]])

# 1. Grafo de conectividad basado en los 12 vecinos más cercanos (k-NN)
grafo_conectividad = kneighbors_graph(X_2d, n_neighbors=12, include_self=False)
print(f'Dimensiones de la matriz de conectividad dispersa: {grafo_conectividad.shape}')
print(f'Elementos no nulos almacenados: {grafo_conectividad.nnz} de {1500*1500} ({grafo_conectividad.nnz / (1500*1500) * 100:.2f}%)')


In [ ]:
# Comparación: AgglomerativeClustering SIN conectividad vs. CON conectividad
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# 1. Sin conectividad: Las distancias euclidianas saltan a través del espacio vacío entre espirales
agg_sin = AgglomerativeClustering(n_clusters=6, linkage='ward')
labels_sin = agg_sin.fit_predict(X_2d)

# 2. Con conectividad: Obliga a seguir la continuidad física de la espiral
agg_con = AgglomerativeClustering(n_clusters=6, linkage='ward', connectivity=grafo_conectividad)
labels_con = agg_con.fit_predict(X_2d)

axes[0].scatter(X_2d[:, 0], X_2d[:, 1], c=labels_sin, cmap='tab10', s=25, edgecolors='none')
axes[0].set_title('Sin Matriz de Conectividad (Euclidiano Ciego):\nFusiona Capas Paralelas de la Espiral', fontsize=11, fontweight='bold')
axes[0].set_xlabel('$X_1$')
axes[0].set_ylabel('$X_2$')

axes[1].scatter(X_2d[:, 0], X_2d[:, 1], c=labels_con, cmap='tab10', s=25, edgecolors='none')
axes[1].set_title('Con Matriz de Conectividad k-NN (k=12):\nRespeta Fielmente la Variedad Curva', fontsize=11, fontweight='bold')
axes[1].set_xlabel('$X_1$')
axes[1].set_ylabel('$X_2$')

plt.suptitle('Poder Topológico del Parámetro connectivity en AgglomerativeClustering', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()


## 2. Benchmark de Escalabilidad: Aceleración por Órdenes de Magnitud

Evaluemos el tiempo de ejecución en función del tamaño del dataset $n$ para demostrar la drástica reducción en tiempo de cómputo que se logra con grafos de adyacencia dispersos.


In [ ]:
# Benchmark de tiempo: n de 1,000 a 6,000 muestras
tamanos_n = [1000, 2000, 3500, 5000]
tiempos_sin = []
tiempos_con = []

for n_sample in tamanos_n:
    X_bench, _ = make_blobs(n_samples=n_sample, n_features=6, centers=8, random_state=42)
    
    # Medición sin conectividad
    t0 = time.perf_counter()
    AgglomerativeClustering(n_clusters=8, linkage='ward').fit(X_bench)
    tiempos_sin.append(time.perf_counter() - t0)
    
    # Medición con conectividad k-NN
    t0 = time.perf_counter()
    knn_graph = kneighbors_graph(X_bench, n_neighbors=15, include_self=False)
    AgglomerativeClustering(n_clusters=8, linkage='ward', connectivity=knn_graph).fit(X_bench)
    tiempos_con.append(time.perf_counter() - t0)

plt.figure(figsize=(9, 5))
plt.plot(tamanos_n, tiempos_sin, 'o--', color='#d62728', lw=2.2, label='AgglomerativeClustering Sin Conectividad (Matriz Densa)')
plt.plot(tamanos_n, tiempos_con, 's-', color='#2ca02c', lw=2.2, label='AgglomerativeClustering Con Grafo k-NN (Matriz Dispersa)')
plt.xlabel('Número de Observaciones (N)', fontsize=11, fontweight='bold')
plt.ylabel('Tiempo de Ejecución (segundos)', fontsize=11, fontweight='bold')
plt.title('Benchmark de Escalabilidad: Aceleración con Restricciones de Conectividad', fontsize=12, fontweight='bold')
plt.legend(fontsize=10.5)
plt.tight_layout()
plt.show()

aceleracion_max = tiempos_sin[-1] / tiempos_con[-1]
print(f'Para N = {tamanos_n[-1]} muestras, la conectividad acelera el algoritmo {aceleracion_max:.1f}x veces.')


## 3. Conclusiones

1. **Superación del Límite $O(n^2)$**: El parámetro `connectivity` transforma el clustering aglomerativo en una técnica escalable a decenas de miles de registros.
2. **Respeto a Variedades No Convexas**: Obligar a que las fusiones sigan las aristas de un grafo $k$-NN permite que el enlace de Ward identifique estructuras espirales y curvas sin requerir DBSCAN.

En el siguiente cuaderno, **05_Aplicacion_Financiera_Hierarchical_Risk_Parity.ipynb**, exploraremos una de las aplicaciones financieras más revolucionarias del clustering jerárquico: el algoritmo **HRP** para construcción de portafolios de inversión robustos.
